# NB03: Recompute deltaG via dGPredictor with OPAM2 pKas

After NB02 updated ModelSEEDDatabase compound TSVs with OPAM2 pKa values,
this notebook recomputes deltaG for all feasible ModelSEED reactions using
the updated pKa values in the Legendre transform.

**Key insight**: The group contribution model (BayesianRidge) predicts
pKa-independent molecular fingerprint energies (`dG_model_only`). Only the
Legendre transform pH/ionic-strength correction (`ddG0_pH_correction`)
depends on pKa. We reuse existing `dG_model_only` values and recompute
only the pH correction using the OPAM2-updated compound cache.

The Legendre transform formula:
```
dG0s = -cumsum([0] + pKas) * R * T * ln(10)
```
A 1 pKa unit change shifts dG by ~5.71 kJ/mol per microspecies.

In [1]:
import sys
import json
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DG_ROOT = Path('/tmp/dGPredictor')
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR = PROJECT_ROOT / 'figures'

sys.path.insert(0, str(DG_ROOT))
sys.path.insert(0, str(DG_ROOT / 'CC'))

from compound import Compound

## 1. Load baseline (Marvin-based) predictions

These were saved from the existing dGPredictor output. Each entry contains
`dG_model_only` (group contribution, pKa-independent) and `ddG0_pH_correction`
(Legendre transform, pKa-dependent).

In [2]:
old_pred_path = DATA_DIR / 'dg_predictions_marvin.json'
with open(old_pred_path) as f:
    old_results = json.load(f)
print(f'Loaded {len(old_results):,} Marvin-based predictions')

example_rxn = list(old_results.keys())[0]
print(f'\nExample ({example_rxn}):')
for k, v in old_results[example_rxn].items():
    print(f'  {k}: {v:.4f}')

Loaded 31,924 Marvin-based predictions

Example (rxn00001):
  dG_mean: -15.7618
  dG_std: 3.6286
  dG_model_only: 23.4553
  ddG0_pH_correction: -39.2171


## 2. Load OPAM2-updated compound cache

The compound cache was rebuilt with OPAM2 pKa values during the
`retrain_modelseed.py` Step 2 (which completed successfully before OOM
on the BayesianRidge training step). Since we only need the Legendre
transform from this cache, the trained model is not required.

In [3]:
cache_path = DG_ROOT / 'CC' / 'data_cc' / 'modelseed_compounds.json.gz'
compound_dict = {}
with gzip.open(cache_path, 'rt', encoding='utf-8') as f:
    for d in json.load(f):
        compound_dict[d['compound_id']] = Compound.from_json_dict(d)
print(f'Loaded {len(compound_dict):,} compounds from OPAM2-updated cache')

stoich_path = DG_ROOT / 'data' / 'modelseed_reaction_stoich.json'
with open(stoich_path) as f:
    rxn_stoich = json.load(f)
print(f'Loaded {len(rxn_stoich):,} reaction stoichiometries')

Loaded 36,847 compounds from OPAM2-updated cache
Loaded 55,999 reaction stoichiometries


## 3. Recompute pH corrections with OPAM2 pKas

For each reaction, keep `dG_model_only` (pKa-independent group contribution)
and recompute `ddG0_pH_correction` using the OPAM2 compound cache's
`transform_pH7()` (Legendre transform).

In [4]:
pH, I, T = 7.0, 0.25, 298.15

new_results = {}
for rxn_id, old in old_results.items():
    if rxn_id not in rxn_stoich:
        continue
    
    model_only = old['dG_model_only']
    old_std = old['dG_std']
    
    ddG0 = 0.0
    for cpd_id, coeff in rxn_stoich[rxn_id].items():
        if cpd_id in compound_dict:
            ddG0 += coeff * compound_dict[cpd_id].transform_pH7(pH, I, T)
    
    new_results[rxn_id] = {
        'dG_mean': model_only + ddG0,
        'dG_std': old_std,
        'dG_model_only': model_only,
        'ddG0_pH_correction': ddG0,
    }

print(f'Recomputed {len(new_results):,} predictions with OPAM2 pKas')

new_pred_path = DATA_DIR / 'dg_predictions_opam2.json'
with open(new_pred_path, 'w') as f:
    json.dump(new_results, f, indent=2)
print(f'Saved to {new_pred_path}')

Recomputed 31,924 predictions with OPAM2 pKas
Saved to /home/freiburgermsu/BERIL-research-observatory/projects/thermodynamic_impact/data/dg_predictions_opam2.json


## 4. Compare old vs new deltaG values

In [5]:
common_rxns = set(old_results.keys()) & set(new_results.keys())
print(f'Reactions in both: {len(common_rxns):,}')

rows = []
for rxn_id in sorted(common_rxns):
    old = old_results[rxn_id]
    new = new_results[rxn_id]
    rows.append({
        'rxn_id': rxn_id,
        'old_dg': old['dG_mean'],
        'new_dg': new['dG_mean'],
        'diff': new['dG_mean'] - old['dG_mean'],
        'abs_diff': abs(new['dG_mean'] - old['dG_mean']),
        'old_model_only': old['dG_model_only'],
        'new_model_only': new['dG_model_only'],
        'old_pH_correction': old['ddG0_pH_correction'],
        'new_pH_correction': new['ddG0_pH_correction'],
        'pH_correction_diff': new['ddG0_pH_correction'] - old['ddG0_pH_correction'],
    })

df_cmp = pd.DataFrame(rows)
df_cmp.to_csv(DATA_DIR / 'dg_comparison.tsv', sep='\t', index=False)
print(f'Saved {len(df_cmp):,} comparisons to data/dg_comparison.tsv')

Reactions in both: 31,924


Saved 31,924 comparisons to data/dg_comparison.tsv


In [6]:
print('deltaG change statistics:')
print(f'  Mean |diff|:   {df_cmp["abs_diff"].mean():.2f} kJ/mol')
print(f'  Median |diff|: {df_cmp["abs_diff"].median():.2f} kJ/mol')
print(f'  Max |diff|:    {df_cmp["abs_diff"].max():.2f} kJ/mol')
print(f'  Std of diff:   {df_cmp["diff"].std():.2f} kJ/mol')
print()

bins = [0, 1, 5, 10, 20, 50, float('inf')]
labels = ['<1', '1-5', '5-10', '10-20', '20-50', '>50']
df_cmp['diff_bin'] = pd.cut(df_cmp['abs_diff'], bins=bins, labels=labels)
print('Distribution of |deltaG change| (kJ/mol):')
for label in labels:
    count = (df_cmp['diff_bin'] == label).sum()
    pct = 100.0 * count / len(df_cmp)
    print(f'  {label:>6s}: {count:6d} ({pct:5.1f}%)')

print()
print('Verification: model-only dG identical (group fingerprints unchanged):')
model_diff = (df_cmp['new_model_only'] - df_cmp['old_model_only']).abs()
print(f'  Max |model_only diff|: {model_diff.max():.10f} kJ/mol')

sign_changed = df_cmp[(df_cmp['old_dg'] * df_cmp['new_dg']) < 0]
print(f'\nSign changes (direction reversal): {len(sign_changed):,} ({len(sign_changed)/len(df_cmp):.1%})')
print(f'  Became favorable (+dG -> -dG):   {((sign_changed["old_dg"] > 0) & (sign_changed["new_dg"] < 0)).sum()}')
print(f'  Became unfavorable (-dG -> +dG): {((sign_changed["old_dg"] < 0) & (sign_changed["new_dg"] > 0)).sum()}')

deltaG change statistics:
  Mean |diff|:   13.35 kJ/mol
  Median |diff|: 0.11 kJ/mol
  Max |diff|:    22716.79 kJ/mol
  Std of diff:   250.17 kJ/mol

Distribution of |deltaG change| (kJ/mol):
      <1:  18658 ( 58.4%)
     1-5:   5634 ( 17.6%)
    5-10:    630 (  2.0%)
   10-20:    184 (  0.6%)
   20-50:   1589 (  5.0%)
     >50:    606 (  1.9%)

Verification: model-only dG identical (group fingerprints unchanged):
  Max |model_only diff|: 0.0000000000 kJ/mol

Sign changes (direction reversal): 840 (2.6%)
  Became favorable (+dG -> -dG):   365
  Became unfavorable (-dG -> +dG): 475


## 5. Visualization

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Histogram of deltaG differences (trimmed to ±100)
ax = axes[0, 0]
trimmed = df_cmp[df_cmp['abs_diff'] <= 100]['diff']
ax.hist(trimmed, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel('deltaG change (kJ/mol)')
ax.set_ylabel('Number of reactions')
ax.set_title(f'Distribution of deltaG changes (N={len(trimmed)}, excl. {len(df_cmp)-len(trimmed)} outliers)')
ax.text(0.02, 0.95, f'Mean: {trimmed.mean():.1f}\nMedian: {trimmed.median():.2f}\nStd: {trimmed.std():.1f}',
        transform=ax.transAxes, verticalalignment='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 2. Parity plot: old vs new dG (trimmed)
ax = axes[0, 1]
mask = (df_cmp['old_dg'].abs() < 500) & (df_cmp['new_dg'].abs() < 500)
sub = df_cmp[mask]
ax.scatter(sub['old_dg'], sub['new_dg'], s=1, alpha=0.3, color='steelblue')
lims = [min(sub['old_dg'].min(), sub['new_dg'].min()),
        max(sub['old_dg'].max(), sub['new_dg'].max())]
ax.plot(lims, lims, 'r--', alpha=0.7, label='y=x')
ax.set_xlabel('Marvin deltaG (kJ/mol)')
ax.set_ylabel('OPAM2 deltaG (kJ/mol)')
ax.set_title(f'deltaG parity (N={len(sub)})')
ax.legend()

# 3. pH correction difference distribution
ax = axes[1, 0]
ph_diff = df_cmp[df_cmp['pH_correction_diff'].abs() <= 100]['pH_correction_diff']
ax.hist(ph_diff, bins=100, color='darkorange', edgecolor='none', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel('pH correction change (kJ/mol)')
ax.set_ylabel('Number of reactions')
ax.set_title('Distribution of Legendre transform correction changes')

# 4. CDF of absolute changes
ax = axes[1, 1]
sorted_abs = np.sort(df_cmp['abs_diff'].values)
cdf = np.arange(1, len(sorted_abs)+1) / len(sorted_abs)
ax.plot(sorted_abs, cdf, color='steelblue', linewidth=2)
ax.set_xlim(0, 100)
ax.set_xlabel('Absolute deltaG change (kJ/mol)')
ax.set_ylabel('Cumulative fraction')
ax.set_title('CDF of absolute deltaG changes')
for thresh in [1, 5, 10, 20]:
    frac = (df_cmp['abs_diff'] <= thresh).mean()
    ax.axhline(frac, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(thresh, color='gray', linestyle=':', alpha=0.5)
    ax.text(thresh+1, frac-0.03, f'{frac:.1%} <= {thresh}', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dg_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/dg_comparison.png')

Saved figures/dg_comparison.png


## 6. Top reactions by deltaG change

In [8]:
print('Top 20 reactions by |deltaG change|:\n')
top20 = df_cmp.nlargest(20, 'abs_diff')[['rxn_id', 'old_dg', 'new_dg', 'diff', 'pH_correction_diff']]
print(top20.to_string(index=False, float_format='%.1f'))

print(f'\n\nSign-change reactions (direction reversal): {len(sign_changed):,}')
print(f'  Sample (first 10):')
sample_sign = sign_changed.nlargest(10, 'abs_diff')[['rxn_id', 'old_dg', 'new_dg', 'diff']]
print(sample_sign.to_string(index=False, float_format='%.1f'))

Top 20 reactions by |deltaG change|:

  rxn_id  old_dg   new_dg     diff  pH_correction_diff
rxn58314 -7605.7 -30322.5 -22716.8            -22716.8
rxn21387 -4459.1 -27166.3 -22707.2            -22707.2
rxn57157 -3287.6 -15294.0 -12006.4            -12006.4
rxn21397  -796.5 -12795.4 -11998.8            -11998.8
rxn59074  5448.7  16855.6  11407.0             11407.0
rxn21394  5055.3  16461.1  11405.8             11405.8
rxn21395  5055.3  16461.1  11405.8             11405.8
rxn21396  2374.2   6531.9   4157.7              4157.7
rxn46297 -1933.7  -5479.1  -3545.4             -3545.4
rxn09021  -620.9  -4078.7  -3457.9             -3457.9
rxn15928    11.3  -3004.7  -3016.0             -3016.0
rxn14340  -375.5  -3390.6  -3015.1             -3015.1
rxn31782    80.9  -2934.2  -3015.1             -3015.1
rxn35377  -375.5  -3390.6  -3015.1             -3015.1
rxn24852   -18.8   2996.0   3014.9              3014.9
rxn24853   -18.8   2996.0   3014.9              3014.9
rxn30960  -176.5   2838.4  

In [9]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
categories = ['No change\n(same sign)', 'Became\nfavorable', 'Became\nunfavorable']
same_sign = len(df_cmp) - len(sign_changed)
pos_to_neg = ((sign_changed['old_dg'] > 0) & (sign_changed['new_dg'] < 0)).sum()
neg_to_pos = ((sign_changed['old_dg'] < 0) & (sign_changed['new_dg'] > 0)).sum()
counts = [same_sign, pos_to_neg, neg_to_pos]
colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax.bar(categories, counts, color=colors, edgecolor='white', linewidth=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count}\n({100*count/len(df_cmp):.1f}%)', ha='center', fontsize=10)
ax.set_ylabel('Number of reactions')
ax.set_title(f'deltaG direction changes: Marvin vs OPAM2 (N={len(df_cmp):,})')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dg_sign_changes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/dg_sign_changes.png')

Saved figures/dg_sign_changes.png
